In [ ]:
import sys
from pathlib import Path

# Racine du projet : le dossier parent de notebooks/
PROJECT_ROOT = Path.cwd().parent.parent
PERIMETER_PATH = PROJECT_ROOT / "data" / "food_perimeter.parquet"
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.cleaning import prepare_data


# Q1

In [ ]:
df = prepare_data(pd.read_parquet(PERIMETER_PATH))
df.head()

In [ ]:
df.info(memory_usage="deep")

In [ ]:
NUTRIENTS = [
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
]

In [ ]:
# category au lieu de str : équivalent de notre table "categories" en SQL permet de donner un id précis puisqu'il n'y a pas beaucoup de valeur différentes.
df["nutriscore_grade"] = df["nutriscore_grade"].astype('category')
df["food_group"] = df["food_group"].astype('category')
# float32 au lieu de float64 : nombre plus petit, moins précis, mais pèse moins dans le fichier
df[NUTRIENTS] = df[NUTRIENTS].astype('float32')

In [ ]:
# Je vérifie si le changement a été fait.
df.info(memory_usage="deep") # 108.7MB contre 137.4MB avant le changement des types.

In [ ]:
import shutil

OUTPUT_PATH = PROJECT_ROOT / "data" / "perimeter_partitioned"
shutil.rmtree(OUTPUT_PATH, ignore_errors=True) # supprimme le dossier et ce qu'il contient
df.to_parquet(OUTPUT_PATH, partition_cols=['food_group']) # partitionne par 'food_group' et créer donc 9 dossiers et 1 fichier dans chacun.

# Q2

In [ ]:
import duckdb
# Pour plus de lisibilité je le renomme en gardant OUTPUT_PATH séparé, au cas où je viens à le changer
SOURCE = PROJECT_ROOT / "data" / "perimeter_partitioned" / "**" / "*.parquet"
duckdb.sql(f"SELECT COUNT(*) FROM '{SOURCE}'").show()

In [ ]:
# Requête nombre de produits, de marques, de rayons
duckdb.sql(f"SELECT COUNT(*) AS products, COUNT(DISTINCT brand) AS brands, COUNT(DISTINCT food_group) as rayons FROM '{SOURCE}'").show()

In [ ]:
# Requête pour savoir si un produit n'a ni de rayon, ni de catégorie
duckdb.sql(f"SELECT COUNT(*) AS orphelins FROM '{SOURCE}' WHERE food_group IS NULL AND categories_tags IS NULL").show()

In [ ]:
# Requête pour les 11 marques les plus présentes
duckdb.sql(f"""
    SELECT brand, COUNT(*) AS nbre
    FROM '{SOURCE}'
    WHERE brand IS NOT NULL
    GROUP BY brand
    ORDER BY nbre DESC
    LIMIT 11
""").show()


In [ ]:
# Requête pour le % de présence du Nutriscore dans chaque rayons
duckdb.sql(f"""
    SELECT food_group, COUNT(*) AS tot, COUNT(nutriscore_grade)                           AS nb,
    ROUND(COUNT(nutriscore_grade) * 100.0 / COUNT(*), 1) AS cpl
    FROM '{SOURCE}'
    GROUP BY food_group
    ORDER BY tot DESC
""").show()


In [ ]:
# Requête de la présence en % du Nutriscore dans les 10 premières catégories (décroissante).
duckdb.sql(f"""
    SELECT tag, COUNT(*) AS tot, COUNT(nutriscore_grade) AS nb,
    ROUND(COUNT(nutriscore_grade) * 100.0 / COUNT(*), 1) AS cpl
    FROM (
        SELECT UNNEST(categories_tags) AS tag, nutriscore_grade
        FROM '{SOURCE}'
    )
    GROUP BY tag
    ORDER BY tot DESC
    LIMIT 10
""").show()


# Q3

In [ ]:
# On créer le pathing vers le CSV
CSV_PATH = PROJECT_ROOT / "data" / "permiter.csv"

In [ ]:
# Exporter le DF en CSV
df.to_csv(CSV_PATH, index=False)

In [ ]:
# Le récupérer
csv = pd.read_csv(CSV_PATH)

# Afficher le nombre de ligne
result = len(csv)
print(result)

In [ ]:
# Afficher la taille en Mo du parquet et du CSV
print(f"CSV     : {CSV_PATH.stat().st_size / 1e6:.0f} Mo")
print(f"Parquet : {sum(f.stat().st_size for f in OUTPUT_PATH.rglob('*.parquet')) / 1e6:.0f} Mo")

In [ ]:
import threading
import time
import psutil

process = psutil.Process()

def measure(code):
    """Lance `code` et renvoie (durée en secondes, pic de mémoire en Mo)."""
    before = process.memory_info().rss
    peak = before
    running = True

    def watch():
        # Permet de modifier la variable sans avoir à en faire une globale
        nonlocal peak
        while running:
            peak = max(peak, process.memory_info().rss)
            time.sleep(0.005)
    
    watcher = threading.Thread(target=watch)
    watcher.start()
    start = time.perf_counter()
    code()
    duration = time.perf_counter() - start
    running = False
    watcher.join()
    return duration, (peak - before) / 1e6


In [ ]:
import psycopg2
from getpass import getpass

# On se connecter à la BDD
conn = psycopg2.connect(
    dbname="nutriscope", host="localhost", port="5432",
    user="postgres", password=getpass("Mot de passe : "),
)
cursor = conn.cursor()

def run_pg(query):
    cursor.execute(query)
    return cursor.fetchall()


In [ ]:
# Requête 1
PG_VOLUMETRIE = """
    SELECT COUNT(*), COUNT(DISTINCT brand_id),
           (SELECT COUNT(*) FROM categories WHERE main)
    FROM products
"""
# Requête 2
PG_COMPLETUDE = """
    SELECT c.name, COUNT(*), COUNT(p.nutriscore_grade)
    FROM products p
    JOIN products_categories pc ON pc.product_id = p.code
    JOIN categories c ON c.id = pc.category_id
    WHERE c.main
    GROUP BY c.name
"""
# Requête 3
PG_SUCRE = """
    SELECT AVG(n.sugars)
    FROM nutriments n
    JOIN products_categories pc ON pc.product_id = n.product_id
    JOIN categories c ON c.id = pc.category_id
    WHERE c.main AND c.name = 'en:beverages'
"""

cases = [
    ("CSV + pandas", "1. Volumétrie", lambda: len(pd.read_csv(CSV_PATH))),
    ("CSV + pandas", "2. Complétude par rayon", lambda: pd.read_csv(CSV_PATH).groupby("food_group")["nutriscore_grade"].agg(["size", "count"])),
    ("CSV + pandas", "3. Sucre des boissons", lambda: pd.read_csv(CSV_PATH).query("food_group == 'en:beverages'")["sugars_100g"].mean()),
    ("PostgreSQL", "1. Volumétrie", lambda: run_pg(PG_VOLUMETRIE)),
    ("PostgreSQL", "2. Complétude par rayon", lambda: run_pg(PG_COMPLETUDE)),
    ("PostgreSQL", "3. Sucre des boissons", lambda: run_pg(PG_SUCRE)),
    ("Parquet + DuckDB", "1. Volumétrie", lambda: duckdb.sql(f"SELECT COUNT(*) FROM '{SOURCE}'").fetchall()),
    ("Parquet + DuckDB", "2. Complétude par rayon", lambda: duckdb.sql(f"SELECT food_group, COUNT(*), COUNT(nutriscore_grade) FROM '{SOURCE}' GROUP BY food_group").fetchall()),
    ("Parquet + DuckDB", "3. Sucre des boissons", lambda: duckdb.sql(f"SELECT AVG(sugars_100g) FROM '{SOURCE}' WHERE food_group = 'en:beverages'").fetchall()),
]


In [ ]:
results = []
for method, query, code in cases:
    measure(code)                        # 1er passage d'échauffement, non compté
    for run in range(3):
        duration, memory = measure(code)
        results.append({"méthode": method, "requête": query,
                        "temps (s)": duration, "mémoire (Mo)": memory})

results = pd.DataFrame(results)
summary = results.groupby(["requête", "méthode"]).median().round(3)


In [ ]:
summary

In [ ]:
results

In [ ]:
import pandas as pd
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

PERIMETER_PATH = PROJECT_ROOT / "data" / "food_perimeter.parquet"
FEATURES_PATH = PROJECT_ROOT / "data" / "features.parquet"

from src.cleaning import prepare_data

df = prepare_data(pd.read_parquet(PERIMETER_PATH))

In [2]:
COLS = [
    'code',
    'product_name',
    'nutriscore_grade',
    'energy-kcal_100g', 
    'fat_100g', 
    'saturated-fat_100g',
    'carbohydrates_100g', 
    'sugars_100g', 
    'fiber_100g', 
    'proteins_100g',
    'salt_100g', 
    'food_group'
]


In [3]:

print(df.shape)
df = df[df['nutriscore_grade'].notna() & ~df['is_incoherent']]
df.shape

(456479, 16)


(421396, 16)

In [4]:
df.columns

Index(['code', 'product_name', 'brands', 'categories_tags', 'nutriscore_grade',
       'energy-kcal_100g', 'fat_100g', 'saturated-fat_100g',
       'carbohydrates_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g',
       'salt_100g', 'food_group', 'brand', 'is_incoherent'],
      dtype='str')

In [5]:
df = df[COLS]

In [6]:
df.shape

(421396, 12)

In [7]:
df = pd.get_dummies(df, columns=['food_group'])
df.shape

(421396, 20)

In [8]:
df.dtypes

code                                         str
product_name                                 str
nutriscore_grade                             str
energy-kcal_100g                         float64
fat_100g                                 float64
saturated-fat_100g                       float64
carbohydrates_100g                       float64
sugars_100g                              float64
fiber_100g                               float64
proteins_100g                            float64
salt_100g                                float64
food_group_en:beverages                     bool
food_group_en:cereals-and-potatoes          bool
food_group_en:composite-foods               bool
food_group_en:fats-and-sauces               bool
food_group_en:fish-meat-eggs                bool
food_group_en:fruits-and-vegetables         bool
food_group_en:milk-and-dairy-products       bool
food_group_en:salty-snacks                  bool
food_group_en:sugary-snacks                 bool
dtype: object

In [9]:
TARGET = "food_group"

In [11]:
NUTRIENTS = [
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
]

In [12]:
df["nutriscore_grade"] = df["nutriscore_grade"].astype('category')
df[NUTRIENTS] = df[NUTRIENTS].astype('float32')

In [13]:
df.dtypes

code                                          str
product_name                                  str
nutriscore_grade                         category
energy-kcal_100g                          float32
fat_100g                                  float32
saturated-fat_100g                        float32
carbohydrates_100g                        float32
sugars_100g                               float32
fiber_100g                                float32
proteins_100g                             float32
salt_100g                                 float32
food_group_en:beverages                      bool
food_group_en:cereals-and-potatoes           bool
food_group_en:composite-foods                bool
food_group_en:fats-and-sauces                bool
food_group_en:fish-meat-eggs                 bool
food_group_en:fruits-and-vegetables          bool
food_group_en:milk-and-dairy-products        bool
food_group_en:salty-snacks                   bool
food_group_en:sugary-snacks                  bool


In [14]:
df.to_parquet(FEATURES_PATH, index=False)

In [ ]:
print("Test de récupération du features.parquet")
df = pd.read_parquet(FEATURES_PATH)
print(f"Nombre de lignes : {len(df)}")
print(f"Nombre de colonnes : {df.shape[1]}")
print(f"Colonnes : {list(df.columns)}")

Test de récupération du features.parquet
Nombre de lignes : 421396
Nombre de colonnes : 20
Nombre de colonnes : ['code', 'product_name', 'nutriscore_grade', 'energy-kcal_100g', 'fat_100g', 'saturated-fat_100g', 'carbohydrates_100g', 'sugars_100g', 'fiber_100g', 'proteins_100g', 'salt_100g', 'food_group_en:beverages', 'food_group_en:cereals-and-potatoes', 'food_group_en:composite-foods', 'food_group_en:fats-and-sauces', 'food_group_en:fish-meat-eggs', 'food_group_en:fruits-and-vegetables', 'food_group_en:milk-and-dairy-products', 'food_group_en:salty-snacks', 'food_group_en:sugary-snacks']
